In [61]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import json

In [87]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# หาหน่วยเลือกตั้งของจังหวัด
rayong = next((p for p in data['provinces'] if p['name'] == 'ระยอง'), None)
# print(f"ระยอง {rayong['total_stations']} หน่วยเลือกตั้ง")
d = dict()
sd2d = dict()
t = 0
# หาหน่วยเลือกตั้งจากรหัส
def find_total_station(area,province):
    for p in data['provinces']:
        if p['name'] == province:
            for a in p['areas']:
                if a['area'] == area:
                    for unit in a['stations']:
                        if unit['district'] not in d:
                            d[unit['district']] = dict()
                        if unit['subdistrict'] not in  d[unit['district']]:
                            d[unit['district']][unit['subdistrict']] = 1
                            sd2d[unit['subdistrict']] = unit['district']
                        else:
                            d[unit['district']][unit['subdistrict']] += 1
    return None

find_total_station(2, 'ลำปาง')
print(d)
print(sd2d)
for k, v in d.items():
    for e, g in v.items():
        t+=g
print(t)

d['นอกเขต'] = dict()
for i in range(1,14):
    sub_dist = f"ชุดที่ {i}"
    sd2d[sub_dist] = 'นอกเขต'
    d['นอกเขต'][sub_dist] = 1
    # all_subdistrict.append(sub_dist)

d['วังเหนือ']['บ้านใหม่'] = 4
d['วังเหนือ']['วังเหนือ'] = 8
d['แจ้ห่ม']['แจ้ห่ม(ในเขต)'] = 6
d['แจ้ห่ม']['แจ้ห่ม(นอกเขต)'] = 8
d['แจ้ห่ม'].pop('แจ้ห่ม', None)
sd2d['บ้านใหม่'] = 'วังเหนือ'
sd2d['แจ้ห่ม(ในเขต)'] = "แจ้ห่ม"
sd2d['แจ้ห่ม(นอกเขต)'] = "แจ้ห่ม"

all_distrcit = []
all_subdistrict = []
district_count = {}
for k,v in d.items():
    all_distrcit.append(k)
    all_subdistrict += v.keys()
    for key in v.keys():
        if k not in district_count:
            district_count[k] = v[key]
        else: district_count[k]+= v[key]

{'เมืองลำปาง': {'บ้านแลง': 12, 'บ้านเสด็จ': 19}, 'งาว': {'หลวงเหนือ': 7, 'หลวงใต้': 8, 'บ้านโป่ง': 12, 'บ้านร้อง': 13, 'ปงเตา': 13, 'นาแก': 6, 'บ้านอ้อน': 8, 'บ้านแหง': 9, 'บ้านหวด': 7, 'แม่ตีบ': 7}, 'แจ้ห่ม': {'แจ้ห่ม': 14, 'บ้านสา': 10, 'ปงดอน': 9, 'แม่สุก': 12, 'เมืองมาย': 7, 'ทุ่งผึ้ง': 8, 'วิเชตนคร': 12}, 'วังเหนือ': {'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}, 'เมืองปาน': {'เมืองปาน': 11, 'บ้านขอ': 14, 'ทุ่งกว๋าว': 16, 'แจ้ซ้อน': 15, 'หัวเมือง': 9}}
{'บ้านแลง': 'เมืองลำปาง', 'บ้านเสด็จ': 'เมืองลำปาง', 'หลวงเหนือ': 'งาว', 'หลวงใต้': 'งาว', 'บ้านโป่ง': 'งาว', 'บ้านร้อง': 'งาว', 'ปงเตา': 'งาว', 'นาแก': 'งาว', 'บ้านอ้อน': 'งาว', 'บ้านแหง': 'งาว', 'บ้านหวด': 'งาว', 'แม่ตีบ': 'งาว', 'แจ้ห่ม': 'แจ้ห่ม', 'บ้านสา': 'แจ้ห่ม', 'ปงดอน': 'แจ้ห่ม', 'แม่สุก': 'แจ้ห่ม', 'เมืองมาย': 'แจ้ห่ม', 'ทุ่งผึ้ง': 'แจ้ห่ม', 'วิเชตนคร': 'แจ้ห่ม', 'ทุ่งฮั้ว': 'วังเหนือ', 'วังเหนือ': 'วังเหนือ', 'วังใต้': 'วังเหนือ', 'ร่องเคาะ': 'วังเ

# Data Extraction

In [60]:
url = "https://www.thaiphc.net/phc/phcadmin/administrator/Report/OSMRP000VHV.php"
init_body = {
    "provinceid": 52,
    "ampurid": "",
    "tambonid": "",
    "search": "ออกรายงาน" 
}

In [53]:
response = requests.post(url=url, data=init_body)
response.encoding = 'tis-620'

In [54]:
if response.status_code == 200:
    html = response.text

In [73]:
district2id = dict()
sub_district2val = dict()

In [74]:
soup = BeautifulSoup(html, "lxml")
for district in all_distrcit:
    option = soup.select_one(f"option:-soup-contains('{district}')")
    if option:
        val = option.get("value")
        district2id[district] = val
    else:
        district2id[district] = -1

In [75]:
all_subdistrict.append("แจ้ห่ม")

In [81]:
for district, id in district2id.items():
    if id == -1:continue
    print(district)
    init_body['ampurid'] = int(id)
    response = requests.post(url=url, data=init_body)
    response.encoding = 'tis-620'
    if response.status_code == 200:
        html = response.text
        soup = BeautifulSoup(html, "lxml")
        tables = soup.select("tr>td.tableb")
        for i, row in enumerate(tables):
            a = row.select_one("a")
            if a and a.text.strip() in all_subdistrict:
                record = tables[i+1:i+6]
                sub_district2val[a.text.strip()] = dict()
                sub_district2val[a.text.strip()]['สมัครเป็นจิตอาสาแล้ว'] = int(record[0].text.strip())
                sub_district2val[a.text.strip()]['จะสมัครเป็นจิตอาส'] = int(record[1].text.strip())
                sub_district2val[a.text.strip()]['ยังไม่พร้อมเป็นจิตอาสา'] = int(record[2].text.strip())
                sub_district2val[a.text.strip()]['ยังไม่ได้เลือกสถานะจิตอาสา'] = int(record[3].text.strip())
                sub_district2val[a.text.strip()]['ยังไม่ได้เลือกสถานะจิตอาสา'] = int(record[4].text.strip())
    else:
        print(f"Something went wrong with code: {response.status_code}")

เมืองลำปาง
งาว
แจ้ห่ม
วังเหนือ
เมืองปาน


In [ ]:
for sd in all_subdistrict:
    if sd not in sub_district2val and "ชุดที่" not in sd and "แจ้ห่ม" not in sd:
        sub_district2val[sd] = dict()
        sub_district2val[sd]['สมัครเป็นจิตอาสาแล้ว'] = -1
        sub_district2val[sd]['จะสมัครเป็นจิตอาส'] = -1
        sub_district2val[sd]['ยังไม่พร้อมเป็นจิตอาสา'] = -1
        sub_district2val[sd]['ยังไม่ได้เลือกสถานะจิตอาสา'] = -1
        sub_district2val[sd]['ยังไม่ได้เลือกสถานะจิตอาสา'] = -1

AttributeError: 'NoneType' object has no attribute 'text'